# Library/Package Requirements

In [ ]:
!pip install pandas numpy scikit-learn mlxtend tqdm Sastrawi

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 209.7/209.7 kB 4.7 MB/s eta 0:00:00


# Import

In [1]:
import pandas as pd
import numpy as np
import re
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory
import warnings
import logging

# Bungkam DeprecationWarning dari internal Jupyter/Python
warnings.filterwarnings("ignore", category=DeprecationWarning)

# Opsional: Bungkam juga log internal Colab yang mengganggu
logging.getLogger("jupyter_client").setLevel(logging.ERROR)

# Config

In [2]:
CONFIG = {
    "URLS": {
        "places": "https://github.com/ray-ahmad/artour_recommender_datasets/raw/main/all_places_exclude_under_review.csv",
        "interactions": "https://github.com/ray-ahmad/artour_recommender_datasets/raw/main/user_interactions.csv"
    },
    "PREPROCESSING": {
        "min_rating_positive": 4.0
    },
    "APRIORI": {
        "min_absolute_support": 3,
        "k_max": 3
    },
    "MCRS": {
        "min_rating_scale": 1.0,
        "max_rating_scale": 5.0,
        "weight_cost": 0.5,
        "weight_benefit": 0.5
    }
}

# Preprocessing

In [3]:
class ARTourDataPreprocessor:
    def __init__(self, config):
        self.config = config
        self.df_places_master = None
        self.df_interactions_master = None
        self._load_and_clean_data()

    def _load_and_clean_data(self):
        print("[1/3] Mengunduh dataset dari URL...")
        df_p = pd.read_csv(self.config["URLS"]["places"])
        df_i = pd.read_csv(self.config["URLS"]["interactions"])

        # --- CLEANING PLACES & NLP MEMOIZATION ---
        print("[2/3] Memulai ekstraksi NLP (Sastrawi) dengan Memoization...")
        df_p['placePrice'] = pd.to_numeric(df_p['placePrice'], errors='coerce').fillna(0)
        df_p['placeRating'] = pd.to_numeric(df_p['placeRating'], errors='coerce').fillna(0)

        # 1. Agregasi Teks
        text_cols = ['placeName', 'placeCategoryName', 'placeDescription', 'placeAddress', 'placeHashtags']
        df_p['raw_combined'] = df_p[text_cols].fillna('').agg(' '.join, axis=1)

        # 2 & 3. Case Folding & RegEx (Hanya a-z dan spasi)
        df_p['clean_text'] = df_p['raw_combined'].str.lower()
        df_p['clean_text'] = df_p['clean_text'].apply(lambda x: re.sub(r'[^a-z\s]', ' ', str(x)))
        df_p['clean_text'] = df_p['clean_text'].apply(lambda x: re.sub(r'\s+', ' ', x).strip())

        # 4 & 5. Memoization Stopword & Stemming
        unique_words = set(" ".join(df_p['clean_text']).split())
        stemmer = StemmerFactory().create_stemmer()
        stopwords = set(StopWordRemoverFactory().get_stop_words())

        word_dict = {}
        for w in unique_words:
            if w not in stopwords:
                word_dict[w] = stemmer.stem(w) # Stemming hanya untuk kata unik
            else:
                word_dict[w] = ""

        # Mapping kamus unik kembali ke kalimat
        df_p['clean_text'] = df_p['clean_text'].apply(
            lambda x: " ".join([word_dict.get(w, "") for w in x.split() if word_dict.get(w, "") != ""]).strip()
        )
        self.df_places_master = df_p

        # --- CLEANING INTERACTIONS ---
        print("[3/3] Membersihkan jejak interaksi pengguna...")
        df_i['createdAt'] = pd.to_datetime(df_i['createdAt'])
        df_i = df_i.sort_values(by='createdAt')
        df_i['value'] = pd.to_numeric(df_i['value'], errors='coerce').fillna(0)

        # Deduplikasi Toggle
        def get_toggle_group(action_type):
            if action_type in ['PLACE_LIKE', 'PLACE_UNLIKE']: return 'TOGGLE_LIKE'
            if action_type in ['PLACE_BOOKMARK', 'PLACE_UNBOOKMARK']: return 'TOGGLE_BOOKMARK'
            if action_type in ['PLACE_DISLIKE', 'PLACE_UNDISLIKE']: return 'TOGGLE_DISLIKE'
            return action_type

        df_i['interaction_group'] = df_i['type'].apply(get_toggle_group)
        df_dedup = df_i.groupby(['userId', 'refId', 'interaction_group']).tail(1).copy()

        # Filter Interaksi Positif
        min_rev = self.config["PREPROCESSING"]["min_rating_positive"]
        valid_actions = ['PLACE_LIKE', 'PLACE_BOOKMARK', 'PLACE_SHARE']
        cond_review = (df_dedup['type'] == 'PLACE_REVIEW') & (df_dedup['value'] >= min_rev)
        cond_others = df_dedup['type'].isin(valid_actions)

        self.df_interactions_master = df_dedup[cond_review | cond_others].sort_values(by='createdAt')
        print("Pra-pemrosesan Selesai!")

    def get_filtered_data(self, min_interactions):
        """Metode ini dipanggil oleh evaluator untuk mengatur Cold-Start user secara dinamis"""
        df_valid = self.df_interactions_master[
            (self.df_interactions_master['refModule'] == 'PLACE') &
            (self.df_interactions_master['refId'].isin(self.df_places_master['placeId']))
        ].copy()

        user_counts = df_valid['userId'].value_counts()
        valid_users = user_counts[user_counts >= min_interactions].index
        df_filtered_int = df_valid[df_valid['userId'].isin(valid_users)].copy()

        return self.df_places_master.copy(), df_filtered_int

# Inisialisasi Singleton Preprocessor
preprocessor = ARTourDataPreprocessor(CONFIG)

[1/3] Mengunduh dataset dari URL...
[2/3] Memulai ekstraksi NLP (Sastrawi) dengan Memoization...
[3/3] Membersihkan jejak interaksi pengguna...
Pra-pemrosesan Selesai!


# Recommender

In [4]:
# ==============================================================================
# SEL 2: MESIN INTI REKOMENDASI (ARTourRecommenderSystem)
# ==============================================================================
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from mlxtend.frequent_patterns import apriori, association_rules

class ARTourRecommenderSystem:
    def __init__(self, df_places, df_int, config, N, K):
        self.config = config
        self.N = N
        self.K = K
        self.df_places = df_places
        self.df_int = df_int

        self.catalog_places = self.df_places['placeId'].tolist()
        self.place_stats = self.df_places.set_index('placeId')[['placePrice', 'placeRating']].to_dict('index')
        self.min_price = self.df_places['placePrice'].min()
        self.max_price = self.df_places['placePrice'].max()

        self.place_id_to_idx = {pid: idx for idx, pid in enumerate(self.catalog_places)}
        self.idx_to_place_id = {idx: pid for idx, pid in enumerate(self.catalog_places)}

        self._prepare_sloo_split()
        self._build_cbf_models()
        self._build_apriori_rules()

    def _prepare_sloo_split(self):
        self.test_data = self.df_int.groupby('userId').tail(1)
        self.train_data = self.df_int.drop(self.test_data.index)

    def _build_cbf_models(self):
        self.vectorizer = TfidfVectorizer()
        self.tfidf_matrix = self.vectorizer.fit_transform(self.df_places['clean_text'])

    def _build_apriori_rules(self):
        basket = self.train_data.groupby(['userId', 'refId'])['refId'].count().unstack().fillna(0)
        basket = (basket > 0)

        num_transactions = len(basket)
        abs_supp = self.config["APRIORI"]["min_absolute_support"]

        # Konversi dinamis Absolute ke Relative Support
        dynamic_min_support = abs_supp / num_transactions if num_transactions > 0 else 0.01

        freq_itemsets = apriori(basket, min_support=dynamic_min_support, use_colnames=True, max_len=self.config["APRIORI"]["k_max"])
        if not freq_itemsets.empty:
            self.rules = association_rules(freq_itemsets, metric="lift", min_threshold=1.0)
        else:
            self.rules = pd.DataFrame()

    def get_apriori_candidates(self, user_basket):
        if self.rules.empty: return []
        candidates = {}
        basket_set = frozenset(user_basket)

        for _, row in self.rules.iterrows():
            if row['antecedents'].issubset(basket_set):
                for item in row['consequents']:
                    if item not in basket_set: # Item Masking (Otomatis)
                        if item not in candidates or row['lift'] > candidates[item]:
                            candidates[item] = row['lift']
        return sorted(candidates.keys(), key=lambda x: candidates[x], reverse=True)

    def get_cbf_candidates(self, user_basket, exclude_cands, needed, anchor_only=False):
        if anchor_only:
            basket_indices = [self.place_id_to_idx[user_basket[-1]]] if user_basket[-1] in self.place_id_to_idx else []
        else:
            basket_indices = [self.place_id_to_idx[p] for p in user_basket if p in self.place_id_to_idx]

        if not basket_indices: return []

        anchor_vector = self.tfidf_matrix[basket_indices].mean(axis=0)
        sim_scores = cosine_similarity(np.asarray(anchor_vector), self.tfidf_matrix).flatten()
        top_indices = sim_scores.argsort()[::-1]

        cbf_cands = []
        for idx in top_indices:
            pid = self.idx_to_place_id[idx]
            if pid not in user_basket and pid not in exclude_cands: # Item Masking
                cbf_cands.append(pid)
                if len(cbf_cands) >= needed: break
        return cbf_cands

    def rerank_mcrs(self, candidates, target_K):
        if not candidates: return []
        scored_cands = []

        c = self.config["MCRS"]
        for p in candidates:
            stats = self.place_stats.get(p)
            if not stats: continue

            norm_price = (stats['placePrice'] - self.min_price) / (self.max_price - self.min_price) if self.max_price > self.min_price else 0
            cost_score = 1.0 - norm_price

            # Normalisasi Rating Mutlak
            norm_rating = (stats['placeRating'] - c["min_rating_scale"]) / (c["max_rating_scale"] - c["min_rating_scale"])
            benefit_score = max(0.0, min(1.0, norm_rating))

            final_score = c["weight_cost"] * cost_score + c["weight_benefit"] * benefit_score
            scored_cands.append((p, final_score))

        scored_cands.sort(key=lambda x: x[1], reverse=True)
        return [x[0] for x in scored_cands[:target_K]]

# Evaluate & Compare

In [5]:
# ==============================================================================
# SEL 3: EVALUATOR KOMPREHENSIF (Ablation + Grid Search MCRS)
# ==============================================================================
from tqdm import tqdm
import numpy as np

class ARTourComprehensiveEvaluator:
    def __init__(self, base_system):
        self.base = base_system

        # Pre-compute Global Popularity dari Training Data
        pop_counts = self.base.train_data['refId'].value_counts()
        self.global_popularity = pop_counts.index.tolist()
        for p in self.base.catalog_places:
            if p not in self.global_popularity:
                self.global_popularity.append(p)

    def _get_popularity_cands(self, basket, target_K):
        cands = []
        for p in self.global_popularity:
            if p not in basket: # Item Masking
                cands.append(p)
                if len(cands) == target_K: break
        return cands

    def _calculate_metrics(self, all_recommended, hits, rr_sum, valid_users):
        hr = hits / valid_users if valid_users > 0 else 0
        mrr = rr_sum / valid_users if valid_users > 0 else 0

        total_items = len(self.base.catalog_places)
        cov = (len(set(all_recommended)) / total_items) * 100 if total_items > 0 else 0

        freq_array = np.zeros(total_items)
        for p in all_recommended:
            if p in self.base.place_id_to_idx:
                freq_array[self.base.place_id_to_idx[p]] += 1

        total_recs = np.sum(freq_array)
        if total_recs > 0 and len(freq_array) > 0:
            n = len(freq_array)
            p_array = freq_array / total_recs
            diff_matrix = np.abs(p_array[:, np.newaxis] - p_array[np.newaxis, :])
            diff_sum = np.sum(diff_matrix)
            mean_p = np.mean(p_array)

            # LOGIKA GINI WAJIB: n dikuadratkan
            gini = diff_sum / (2 * (n ** 2) * mean_p)
        else:
            gini = 0.0

        return hr, mrr, cov, gini

    def run_evaluation(self, nk_combinations):
        models = {}
        # Ekstrak semua nilai K unik dari kombinasi untuk membuat baseline Ablation Study
        unique_k_values = set([k for n, k in nk_combinations])

        # 1. Inisialisasi dictionary untuk Baseline (Ablation Study) per nilai K
        for k in sorted(unique_k_values):
            models[f"1. Popularity Only (K={k})"] = {"hits": 0, "rr": 0, "recs": []}
            models[f"2. Pure Apriori (K={k})"] = {"hits": 0, "rr": 0, "recs": []}
            models[f"3. Pure CBF (K={k})"] = {"hits": 0, "rr": 0, "recs": []}
            models[f"4. Apriori+CBF No MCRS (K={k})"] = {"hits": 0, "rr": 0, "recs": []}

        # 2. Inisialisasi dictionary untuk Hybrid MCRS (Grid Search)
        for N, K in nk_combinations:
            models[f"5. Hybrid MCRS (N={N}, K={K})"] = {"hits": 0, "rr": 0, "recs": []}

        valid_users = 0

        for user_id in tqdm(self.base.test_data['userId'].unique(), desc="Evaluasi Model", leave=False):
            test_row = self.base.test_data[self.base.test_data['userId'] == user_id].iloc[0]
            ground_truth = test_row['refId']

            basket = self.base.train_data[self.base.train_data['userId'] == user_id]['refId'].tolist()
            if not basket: continue

            valid_users += 1
            basket_set = set(basket)

            # Ekstrak Apriori Mentah (sekali saja per user agar hemat komputasi)
            apriori_raw = self.base.get_apriori_candidates(basket)
            apriori_filtered = [p for p in apriori_raw if p not in basket_set]

            # ==============================================================
            # A. EKSEKUSI BASELINE (ABLATION STUDY) UNTUK SETIAP K
            # ==============================================================
            for k in unique_k_values:
                # Model 1: Pop
                m1 = self._get_popularity_cands(basket_set, target_K=k)
                # Model 2: Apriori
                m2 = apriori_filtered[:k]
                # Model 3: CBF
                m3 = self.base.get_cbf_candidates(basket, exclude_cands=[], needed=k, anchor_only=True)
                # Model 4: Apriori + CBF Padding (Tanpa MCRS)
                m4 = list(m2)
                if len(m4) < k:
                    m4.extend(self.base.get_cbf_candidates(basket, m4, needed=k-len(m4), anchor_only=False))

                # Pencatatan Baseline
                baselines = {
                    f"1. Popularity Only (K={k})": m1, f"2. Pure Apriori (K={k})": m2,
                    f"3. Pure CBF (K={k})": m3, f"4. Apriori+CBF No MCRS (K={k})": m4
                }
                for name, cands in baselines.items():
                    models[name]["recs"].extend(cands)
                    if ground_truth in cands:
                        models[name]["hits"] += 1
                        models[name]["rr"] += (1.0 / (cands.index(ground_truth) + 1))

            # ==============================================================
            # B. EKSEKUSI HYBRID MCRS (GRID SEARCH N & K)
            # ==============================================================
            for N, K in nk_combinations:
                name = f"5. Hybrid MCRS (N={N}, K={K})"

                # Over-generation N
                hybrid_cands = list(apriori_filtered[:N])
                if len(hybrid_cands) < N:
                    hybrid_cands.extend(self.base.get_cbf_candidates(basket, hybrid_cands, N-len(hybrid_cands), False))

                # Truncation K
                m5 = self.base.rerank_mcrs(hybrid_cands, target_K=K)

                models[name]["recs"].extend(m5)
                if ground_truth in m5:
                    models[name]["hits"] += 1
                    models[name]["rr"] += (1.0 / (m5.index(ground_truth) + 1))

        # Hitung kalkulasi akhir metrik
        results = {}
        for m_name, data in models.items():
            results[m_name] = self._calculate_metrics(data["recs"], data["hits"], data["rr"], valid_users)

        return results, valid_users


# ==============================================================================
# AUTOMATIC RUNNER: ABLATION + GRID SEARCH
# ==============================================================================
if __name__ == "__main__":

    # -------------------------------------------------------------------------
    # PARAMETER EKSPERIMEN (Sesuai Permintaan Anda)
    # -------------------------------------------------------------------------
    MIN_INTERACTIONS_LIST = [2, 3]
    NK_SCENARIOS = [
        (5, 5),   # MCRS sebagai Shuffler
        (10, 5),  # Over-generation moderat
        (10, 10), # MCRS sebagai Shuffler di K tinggi
        (15, 10), # Over-generation moderat
        (20, 10)  # Over-generation maksimal
    ]

    for min_int in MIN_INTERACTIONS_LIST:
        print("\n" + "="*95)
        print(f"{'EVALUASI KOMPREHENSIF | COLD-START (Min Interaksi = ' + str(min_int) + ')':^95}")
        print("="*95)

        # Panggil data dari Preprocessor (Sel 1)
        df_p, df_i = preprocessor.get_filtered_data(min_interactions=min_int)

        # Init Sistem (Parameter N, K statis disini diabaikan karena ditimpa evaluator)
        sys_eval = ARTourRecommenderSystem(df_p, df_i, CONFIG, N=20, K=10)
        evaluator = ARTourComprehensiveEvaluator(sys_eval)

        # Jalankan
        results, valid_users = evaluator.run_evaluation(NK_SCENARIOS)

        print(f"Total Test Users Valid: {valid_users} users")
        print("-" * 95)
        print(f"{'Arsitektur Model':<35} | {'Hit Rate':<12} | {'MRR':<12} | {'Coverage (%)':<12} | {'Gini Index':<10}")
        print("-" * 95)

        # Sortir hasil agar rapi berdasarkan nama
        for name in sorted(results.keys()):
            hr, mrr, cov, gini = results[name]
            # Pisahkan visual antara Baseline dan MCRS Hybrid
            if name.startswith("5.") and "(N=5, K=5)" in name:
                print("-" * 95)
            print(f"{name:<35} | {hr:<12.4f} | {mrr:<12.4f} | {cov:<12.2f} | {gini:<10.4f}")

        print("="*95)


                    EVALUASI KOMPREHENSIF | COLD-START (Min Interaksi = 2)                     


Total Test Users Valid: 106 users
-----------------------------------------------------------------------------------------------
Arsitektur Model                    | Hit Rate     | MRR          | Coverage (%) | Gini Index
-----------------------------------------------------------------------------------------------
1. Popularity Only (K=10)           | 0.0755       | 0.0230       | 3.61         | 0.9783    
1. Popularity Only (K=5)            | 0.0472       | 0.0197       | 2.47         | 0.9884    
2. Pure Apriori (K=10)              | 0.0755       | 0.0293       | 3.98         | 0.9697    
2. Pure Apriori (K=5)               | 0.0472       | 0.0255       | 3.80         | 0.9750    
3. Pure CBF (K=10)                  | 0.0283       | 0.0090       | 59.96        | 0.6617    
3. Pure CBF (K=5)                   | 0.0189       | 0.0079       | 40.23        | 0.7638    
4. Apriori+CBF No MCRS (K=10)       | 0.0943       | 0.0316       | 55.03        | 0.7416    
4. Apriori+CBF No MCRS

Total Test Users Valid: 40 users
-----------------------------------------------------------------------------------------------
Arsitektur Model                    | Hit Rate     | MRR          | Coverage (%) | Gini Index
-----------------------------------------------------------------------------------------------
1. Popularity Only (K=10)           | 0.1750       | 0.0559       | 3.61         | 0.9741    
1. Popularity Only (K=5)            | 0.1000       | 0.0458       | 2.47         | 0.9851    
2. Pure Apriori (K=10)              | 0.1750       | 0.0559       | 3.98         | 0.9693    
2. Pure Apriori (K=5)               | 0.1250       | 0.0496       | 3.80         | 0.9761    
3. Pure CBF (K=10)                  | 0.0500       | 0.0208       | 38.33        | 0.7465    
3. Pure CBF (K=5)                   | 0.0500       | 0.0208       | 21.82        | 0.8465    
4. Apriori+CBF No MCRS (K=10)       | 0.2000       | 0.0587       | 22.20        | 0.9070    
4. Apriori+CBF No MCRS 